# 🔧 Notebook 02: Feature Engineering
## Trích xuất đặc trưng: Season, Binning, Scaling


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_config, resolve_path
from src.features.builder import FeatureBuilder

config = load_config()
parquet_path = resolve_path(config['data']['cleaned_parquet'])
df_clean = pd.read_parquet(parquet_path)
print(f'Loaded: {df_clean.shape}')


## 1. Chạy FeatureBuilder


In [ ]:
builder = FeatureBuilder(config)
df = builder.run(df_clean)
builder.save(df, fmt='parquet')
print(f'\nShape sau FE: {df.shape}')
print(f'Columns mới: {[c for c in df.columns if c not in df_clean.columns]}')


## 2. Phân bố Season


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df['Season'].value_counts().plot(kind='bar', ax=ax,
     color=['#4CAF50', '#FF9800', '#9C27B0', '#2196F3'])
ax.set_title('Phân bố theo Mùa', fontsize=14, fontweight='bold')
ax.set_ylabel('Số mẫu')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Kiểm tra Binning


In [ ]:
print('Binning Temperature:')
print(df['Temp_Bin'].value_counts())
print('\nBinning Humidity:')
print(df['Humidity_Bin'].value_counts())
print('\nBinning Wind Speed:')
print(df['Wind_Bin'].value_counts())


## 4. Nhiệt độ theo Mùa


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for season in ['Spring', 'Summer', 'Autumn', 'Winter']:
    subset = df[df['Season'] == season]
    ax.hist(subset['Temperature (C)'], bins=50, alpha=0.5, label=season)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Tần suất', fontsize=12)
ax.set_title('Phân bố Nhiệt độ theo Mùa', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Xu hướng nhiệt độ theo thời gian


In [ ]:
df_ts = df.set_index('Formatted Date')['Temperature (C)'].resample('M').mean()
plt.figure(figsize=(16, 5))
plt.plot(df_ts.index, df_ts.values, linewidth=1.5, color='#2196F3')
plt.xlabel('Thời gian')
plt.ylabel('Nhiệt độ TB (°C)')
plt.title('Xu hướng Nhiệt độ Trung bình Hàng tháng (2006-2016)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Kiểm tra Scaling


In [ ]:
scaled_cols = [c for c in df.columns if c.endswith('_scaled')]
print(f'Scaled columns: {scaled_cols}')
print(f'\nMean (should ≈ 0):')
print(df[scaled_cols].mean().round(4))
print(f'\nStd (should ≈ 1):')
print(df[scaled_cols].std().round(4))
